# Exercise: Getting Comfortable with Our Open Lakehouse

This first exercise is setup to make sure that your container environment correctly started up and that
Apache Iceberg tables are accessible from multiple engines.

## The Tools

- **Apache Iceberg**: Open table format — a table format adds database-like guarantees (schemas, transactions, versioning) on top of files in object storage, so you get the reliability of a database without being locked into a single engine
- **MinIO**: S3-compatible object storage
- **Polaris**: Iceberg REST Compatible Catalog
- **Spark**: Distributed Analytics Engine
- **Trino**: Distributed Analytics Engine

## Explore the Environment


### Jupyter Notebooks

**You're here!**

Jupyter provides an interactive environment for running Spark and Python code. All the lesson exercises are stored in the `/notebooks` directory.

**What to explore:**
- The file browser on the left shows all available notebooks
- Each notebook corresponds to an exercise in the course
- You can create new notebooks to experiment with your own queries or code
- The integrated terminal allows you to run shell commands if needed

### Polaris Catalog

**URL:** Polaris UI is not yet released

Polaris is a REST catalog that manages all the metadata for your Iceberg tables. It keeps track of schemas, snapshots (immutable records of the table's state after each write), and table locations without storing the actual data.

**What to explore:**
- TBD

### Spark Configuration

**Configuration File:** `/home/jovyan/.sparkconf/spark-defaults.conf`

The Spark Session configuration is automatically generated by docker-compose and is placed in the above file to wire together all of the components
of this Lakehouse as well as download necessary dependencies for Spark to work with Iceberg. The code below opens that file, reads it into the config variable, and then prints that variable for you to see.

**Key packages included:**
- **Iceberg Spark Runtime** - Iceberg Plugin Integration and Iceberg REST Catalog Client
- **AWS Bundle** - Provides S3 Client for MinIO
- **Catalog Configuration** - Points Spark to the Polaris REST catalog
- **S3 Credentials** - Authenticates with MinIO storage

In [ ]:
with open('/home/jovyan/.sparkconf/spark-defaults.conf', 'r') as f:
    config = f.read()
    print(config)

## Initialize Spark Session

Now let's start coding! We'll create a Spark session that connects to our pre-configured Iceberg environment. As you can see in the printed result, we are using the Polaris catalog.

In [ ]:
from pyspark.sql import SparkSession

# All configuration is loaded from spark-defaults.conf
spark = SparkSession.builder \
    .appName("OpenLakehouse") \
    .getOrCreate()

print(f" Spark {spark.version} initialized!")
print(f" Default Catalog: {spark.conf.get('spark.sql.defaultCatalog', 'spark_catalog')}")

## Create Your First Iceberg Table

Let's create a simple table to demonstrate the lakehouse in action. We'll create a namespace (like a database schema) and a table within it.

In [ ]:


# Create namespace (like a schema in traditional databases)
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.demo")
print(" Namespace 'demo' created!")

# Drop and recreate so re-running the notebook starts fresh
spark.sql("DROP TABLE IF EXISTS polaris.demo.employees")

# Create an Iceberg table
# Format version 3 is the latest Iceberg spec and enables features like initial
# column defaults. Earlier versions (1 and 2) are still in wide use.
spark.sql("""
    CREATE TABLE polaris.demo.employees (
        id INT,
        name STRING,
        department STRING
    ) USING iceberg
    TBLPROPERTIES (
        'format-version' = '3'
    )
""")
print(" Table 'employees' created!")

## Insert Sample Data

Now with our table created and our initial schema set, let's insert a series of data values into this table.

In [ ]:
spark.sql("""
    INSERT INTO polaris.demo.employees VALUES
    (1, 'Alice', 'Engineering'),
    (2, 'Bob', 'Sales'),
    (3, 'Charlie', 'Marketing'),
    (4, 'Diana', 'Engineering')
""")
print(" Sample data inserted!")

## Query with Spark

Now let's confirm that the data is in the table and read the data back using Spark SQL.

In [ ]:
spark.sql("""
    SELECT * FROM polaris.demo.employees
""").show()

In [ ]:
spark.sql("""
    SELECT department, COUNT(*) as employee_count
    FROM polaris.demo.employees
    GROUP BY department
    ORDER BY employee_count DESC
""").show()

### Explore the Spark UI

**URL:** http://localhost:4040

The Spark UI provides detailed information about your Spark jobs:

**What you can see:**
- **Jobs tab**: Overview of all Spark jobs executed
- **Stages tab**: Breakdown of each job into stages and tasks
- **SQL tab**: Query plans and execution details for Spark SQL operations
- **Environment tab**: Configuration settings and runtime properties

This is useful for understanding query performance and debugging issues.

## Query with Trino

One of the key benefits of Apache Iceberg is **engine interoperability**. Traditional systems like MySQL or Hive-managed tables couple the storage layout to a specific engine. Iceberg defines the table structure in open metadata files that any engine can read, so the data is truly engine-independent.

Let's connect to Trino and query the exact same data. Trino is a distributed SQL query engine designed for analytics. It's completely separate from Spark, but both can read/write the same Iceberg tables. Trino is accessible via JDBC which we are using below.

In [ ]:
from trino.dbapi import connect
import pandas as pd
import time

# Connect to Trino (with retry for container startup timing)
max_retries = 3
for attempt in range(max_retries):
    try:
        conn = connect(
            host='trino',
            port=8080,
            user='trino',
            catalog='polaris',
            schema='demo'
        )

        cursor = conn.cursor()

        # Query the same table we created with Spark!
        cursor.execute("SELECT * FROM employees")
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]

        # Display results
        df = pd.DataFrame(rows, columns=columns)
        print(" Querying via Trino:\n")
        print(df.to_string(index=False))

        cursor.close()
        conn.close()
        break
    except Exception as e:
        if attempt < max_retries - 1:
            print(f"Trino connection attempt {attempt + 1} failed, retrying in 5s...")
            time.sleep(5)
        else:
            raise

### Explore the Trino UI

**URL:** http://localhost:8080  
**Credentials:** admin

The Trino UI shows query execution and cluster status:

**What you can see:**
- **Query details**: Click any query to see execution plan, data scanned, and performance metrics. Make sure to select *finished* in the filter to see the above query
- **Cluster overview**: Worker nodes and resource utilization
- **Live query monitoring**: Real-time progress of running queries

The UI is great for monitoring query performance and understanding how Trino processes your SQL.

## Explore MinIO

Now that we've created tables and inserted data, let's explore MinIO to see the actual files that were created!

**URL:** http://localhost:9001/browser/warehouse/demo/employees  
**Credentials:** admin / password

MinIO is an S3-compatible object storage where all data and metadata files are stored.

**What to explore:**
- Click into the `data/` folder to see the actual Parquet data files
- Click into the `metadata/` folder to see Iceberg Metadata Json, Manifest and Manifest List files

Iceberg's metadata forms a tree: **snapshot → manifest list → manifests → data files**. Manifest files track individual data files along with their partition values and column-level statistics. Manifest lists group manifests together and are what snapshots point to. This layered structure is what enables efficient query planning — engines can skip irrelevant files without ever opening them.

## Congrats!

This concludes your first exercise. Now, you've learned to set up your Open Lakehouse by configuring a catalog and storage object and executing queries on them with engines like Trino and Apache Spark.